# Lab 2 – Task 2.1: exploring the raw downloads

This notebook calls each extractor, transforms the saved file into the normalized table
(`country_iso3, indicator_code, year, sex, value`) and inspects the result, with a focus
on missing values.

Run it from the project root folder's virtual environment (the first cell moves to the project root).

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from etl.config import INDICATORS
from etl.extract_owid import download_owid_chart
from etl.extract_who import download_who_indicator
from etl.extract_worldbank import download_worldbank_countries, download_worldbank_indicator
from etl.transform_owid import transform_owid_chart_file
from etl.transform_who import transform_who_file
from etl.transform_worldbank import (
    transform_worldbank_country_file,
    transform_worldbank_indicator_file,
)

## 1. Extract: download every indicator listed in `etl/config.py`

In [2]:
DOWNLOAD = {
    "who": download_who_indicator,
    "world_bank": download_worldbank_indicator,
    "owid": download_owid_chart,
}
TRANSFORM = {
    "who": transform_who_file,
    "world_bank": transform_worldbank_indicator_file,
    "owid": transform_owid_chart_file,
}

countries_path, _ = download_worldbank_countries()
paths = {}
for indicator in INDICATORS:
    path, url = DOWNLOAD[indicator["source"]](indicator["code"])
    paths[indicator["code"]] = path
    print(f"{indicator['code']:<24} {path.name}")

SP.POP.65UP.TO.ZS        SP.POP.65UP.TO.ZS_2000-2024.json


NY.GDP.PCAP.CD           NY.GDP.PCAP.CD_2000-2024.json
SH.MED.BEDS.ZS           SH.MED.BEDS.ZS_2000-2024.json


GHED_CHE_pc_US_SHA2011   GHED_CHE_pc_US_SHA2011_2000-2024.json


GHED_CHEGDP_SHA2011      GHED_CHEGDP_SHA2011_2000-2024.json


GHED_OOPSCHE_SHA2011     GHED_OOPSCHE_SHA2011_2000-2024.json


WHOSIS_000001            WHOSIS_000001_2000-2024.json


WHOSIS_000002            WHOSIS_000002_2000-2024.json


life-expectancy          life-expectancy_full.csv


## 2. Transform: one normalized table per indicator

In [3]:
frames = {
    i["code"]: TRANSFORM[i["source"]](paths[i["code"]], i["code"]) for i in INDICATORS
}
countries = transform_worldbank_country_file(countries_path)
print(f"{len(countries)} countries (aggregates removed), {countries['region'].nunique()} regions")
countries.head()

217 countries (aggregates removed), 7 regions


,iso3,iso2,name,region,income_group
0,ABW,AW,Aruba,Latin America & Caribbean,High income
1,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
2,AGO,AO,Angola,Sub-Saharan Africa,Lower middle income
3,ALB,AL,Albania,Europe & Central Asia,Upper middle income
4,AND,AD,Andorra,Europe & Central Asia,High income


## 3. Inspect one indicator per source

### World Bank – population ages 65 and above

In [4]:
wb = frames["SP.POP.65UP.TO.ZS"]
display(wb.head())
wb.info()
wb.isna().sum()

,country_iso3,indicator_code,year,sex,value
0,AFE,SP.POP.65UP.TO.ZS,2024,None,3.330960
1,AFE,SP.POP.65UP.TO.ZS,2023,None,3.286303
2,AFE,SP.POP.65UP.TO.ZS,2022,None,3.246111
3,AFE,SP.POP.65UP.TO.ZS,2021,None,3.216573
4,AFE,SP.POP.65UP.TO.ZS,2020,None,3.192169


<class 'pandas.DataFrame'>
RangeIndex: 6650 entries, 0 to 6649
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    6650 non-null   str    
 1   indicator_code  6650 non-null   str    
 2   year            6650 non-null   str    
 3   sex             0 non-null      object 
 4   value           6625 non-null   float64
dtypes: float64(1), object(1), str(3)
memory usage: 259.9+ KB


country_iso3         0
indicator_code       0
year                 0
sex               6650
value               25
dtype: int64

### WHO – life expectancy at birth (split by sex)

In [5]:
who = frames["WHOSIS_000001"]
display(who.head())
who.info()
display(who["sex"].value_counts())
who.isna().sum()

,country_iso3,indicator_code,year,sex,value
0,MKD,WHOSIS_000001,2019,SEX_BTSX,76.3
1,NAM,WHOSIS_000001,2000,SEX_MLE,52.4
2,GEO,WHOSIS_000001,2005,SEX_MLE,68.1
3,FRA,WHOSIS_000001,2010,SEX_BTSX,81.2
4,ISL,WHOSIS_000001,2020,SEX_MLE,81.1


<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    13320 non-null  str    
 1   indicator_code  13320 non-null  str    
 2   year            13320 non-null  int64  
 3   sex             13320 non-null  str    
 4   value           13320 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 520.4 KB


sex
SEX_BTSX    4440
SEX_MLE     4440
SEX_FMLE    4440
Name: count, dtype: int64

country_iso3      0
indicator_code    0
year              0
sex               0
value             0
dtype: int64

### Our World in Data – life expectancy (long series)

In [6]:
owid = frames["life-expectancy"]
display(owid.head())
owid.info()
print("years:", owid["year"].min(), "-", owid["year"].max())
owid.isna().sum()

,country_iso3,indicator_code,year,sex,value
0,AFG,life-expectancy,1950,None,28.1563
1,AFG,life-expectancy,1951,None,28.5836
2,AFG,life-expectancy,1952,None,29.0138
3,AFG,life-expectancy,1953,None,29.4521
4,AFG,life-expectancy,1954,None,29.6975


<class 'pandas.DataFrame'>
RangeIndex: 21565 entries, 0 to 21564
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    20217 non-null  str    
 1   indicator_code  21565 non-null  str    
 2   year            21565 non-null  int64  
 3   sex             0 non-null      object 
 4   value           21565 non-null  float64
dtypes: float64(1), int64(1), object(1), str(2)
memory usage: 842.5+ KB
years: 1543 - 2023


country_iso3       1348
indicator_code        0
year                  0
sex               21565
value                 0
dtype: int64

## 4. Share of missing values per indicator

`sex` is left out: it is empty on purpose for sources without a sex split (the cleaning step turns it into `BTSX`).

In [7]:
summary = pd.DataFrame(
    [
        {
            "indicator": code,
            "source": next(i["source"] for i in INDICATORS if i["code"] == code),
            "rows": len(df),
            "missing value": df["value"].isna().sum(),
            "missing value %": round(100 * df["value"].isna().mean(), 1),
            "missing country code": df["country_iso3"].isna().sum()
            + (df["country_iso3"].astype("string").str.strip() == "").sum(),
            "first year": df["year"].astype(int).min(),
            "last year": df["year"].astype(int).max(),
        }
        for code, df in frames.items()
    ]
)
summary

,indicator,source,rows,missing value,missing value %,missing country code,first year,last year
0,SP.POP.65UP.TO.ZS,world_bank,6650,25,0.4,125,2000,2024
1,NY.GDP.PCAP.CD,world_bank,6650,208,3.1,125,2000,2024
2,SH.MED.BEDS.ZS,world_bank,6650,2995,45.0,125,2000,2024
3,GHED_CHE_pc_US_SHA2011,who,4585,0,0.0,0,2000,2023
4,GHED_CHEGDP_SHA2011,who,4588,0,0.0,0,2000,2023
5,GHED_OOPSCHE_SHA2011,who,4588,0,0.0,0,2000,2023
6,WHOSIS_000001,who,13320,0,0.0,0,2000,2023
7,WHOSIS_000002,who,13320,0,0.0,0,2000,2023
8,life-expectancy,owid,21565,0,0.0,1348,1543,2023


## 5. Notes

- **Hospital beds (`SH.MED.BEDS.ZS`)** has by far the most missing values (about 45%): many
  countries do not report it every year. It is kept, but comparisons of beds between countries
  will only work for the years that were reported.
- **GDP per capita** misses about 3% of the values and **population 65+** under 1%. The World Bank
  files also contain regions and income groups (World, Euro area, High income, ...), which are
  not countries and are removed during cleaning; 125 rows per indicator (the income groups) have
  no ISO3 code at all.
- The **WHO** files have no missing values: WHO only returns rows that have a value. Life
  expectancy and HALE come split by sex (`SEX_BTSX`, `SEX_MLE`, `SEX_FMLE`), so they have three
  times more rows.
- The **OWID** file has no missing values but covers years long before 2000 and also regions
  without an ISO3 code (for example continents), so most of its rows are dropped by the year window
  and the country filter during cleaning.
- **Years:** the World Bank already has 2024 values for population 65+ and GDP per capita, but not
  for hospital beds (latest 2023). The WHO and OWID data end in 2023, so 2023 is the latest year
  that all indicators share.